In [0]:
dbutils.widgets.text("catalog","dev")
catalog = dbutils.widgets.get("catalog")

In [0]:
from pyspark.sql import functions as F

In [0]:
df_tranc =spark.read.table(f"{catalog}.bronze.bronze_transactions")

In [0]:
df_accounts = spark.read.table(f"{catalog}.silver.silver_accounts")

In [0]:
df_cleaned_tranc = df_tranc.join(df_accounts.select('account_id'), on='account_id', how='left_semi') \
    .withColumn("transaction_date", F.to_date(F.col("transaction_date"), "yyyy-MM-dd")) \
        .withColumn("transaction_timestamp", F.to_timestamp(F.col("transaction_timestamp"), "yyyy-MM-dd HH:mm:ss")) \
            .withColumn("created_at", F.to_timestamp(F.col("created_at"), "yyyy-MM-dd HH:mm:ss")) \
                .withColumn("amount", F.col("amount").cast("float")) \
                    .withColumn("transaction_type", F.lower(F.trim(F.col("transaction_type")))) \
                        .withColumn("transaction_channel", F.lower(F.trim(F.col("transaction_channel")))) \
                            .withColumn("currency", F.upper(F.trim(F.col("currency")))) \
                                .withColumn("transaction_status", F.lower(F.trim(F.col("transaction_status")))) \
                                    .fillna({"merchant_name": "Unknown",
                                             "merchant_category": "Unknown"}) \
                                                 .dropDuplicates(["transaction_id"]) \
                                                     .dropna(subset=['transaction_id', 'account_id','transaction_date','transaction_timestamp','transaction_type','transaction_channel','amount','currency','transaction_status','reference_number','created_at',])

In [0]:
df_cleaned_tranc.write.mode("overwrite").saveAsTable(f"{catalog}.silver.silver_transactions")